In [1]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

The function `logging` is used as a *decorator*.  It takes a function 
`f` as its argument and returns a new function `logged_f` that returns 
the same result as the function `f`, but additionally it prints its arguments 
before the function is called and when the function returns, both the function 
call and the result is printed.

The *decorator* `logging` is useful for debugging.

In [36]:
function logging<F extends (...args: any[]) => any>(f: F): F {
  const logged_f = ((...args: Parameters<F>): ReturnType<F> => {
    //console.log(`${f.name}${JSON.stringify(args)}`); *uncomment to enable logging
    const r = f(...args);
    //console.log(`${f.name}${JSON.stringify(args)} = ${r}`);
    return r;
  }) as F;
  return logged_f;
}


# An Array-Based Implementation of Quick-Sort

The function $\texttt{swap}(L, x, y)$ swaps the elements at index $x$ and $y$ in $L$.

In [37]:
function swap(L: number[], i: number, j: number): void {
  const temp = L[i];
  L[i] = L[j];
  L[j] = temp;
}

The function $\texttt{partition}(\texttt{start}, \texttt{end}, L)$ returns an index $m$ into the array $L$ and 
regroups the elements of $L$ such that after the function returns the following holds:
 
  - $\forall i \in \{\texttt{start}, \cdots, m-1\} : L[i] \leq L[m]$,
  - $\forall i \in \{ m+1, \cdots, \texttt{end} \}  : L[m] <    L[i]$,
  - $L[m] = \texttt{pivot}$.
  
Here, `pivot` is the element that is at the index `end` at the time of the invocation 
of the function, i.e. we have

  - $L[\texttt{end}] = \texttt{pivot}$
  
at invocation time.
  
The for-loop of `partition` maintains the following invariants:

 - $\forall i \in \{\texttt{start}, \cdots, \texttt{left} \} : L[i] \leq \texttt{pivot}$,
 - $\forall i \in \{\texttt{left}+1, \cdots, \texttt{idx}-1\} : \texttt{pivot} < L[i]$,
 - $L[\texttt{end}] = \texttt{pivot}$.

These invariants are depicted below:

![Invariants for partitioning](lomuto.png)

This algorithm has been suggested by *Nico Lomuto*.  It is not the most efficient implementation of `partition`, but
it is easier to understand than the algorithm given by *Tony Hoare* that uses two separate loops.

In [38]:
const partition = logging(function partition(start: number, end: number, L: number[]): number {
  const pivot = L[end];
  let left = start - 1;
  for (let idx = start; idx < end; idx++) {
    if (L[idx] <= pivot) {
      left += 1;
      swap(L, left, idx);
    }
  }
  swap(L, left + 1, end);
  return left + 1;
});

The function `quickSort(start, end, L)` sorts the subarray `L[start:end+1]` in place.

In [39]:
function quickSort(start: number, end: number, L: number[]): void {
  if (end <= start) return; // nothing to do, single element
  const m = partition(start, end, L);
  quickSort(start, m - 1, L);
  quickSort(m + 1, end, L);
}

The function $\texttt{sort}(L)$ sorts the array $L$ in place.

In [40]:
function sort(L: number[]): void {
  quickSort(0, L.length - 1, L);
}

## Testing

In [41]:
function demo() {
  const L: number[] = Array.from({ length: 19 }, () => Math.floor(Math.random() * 99) + 1);
  console.log("L =", L);
  sort(L);
  console.log("L =", L);
}

In [42]:
demo();

L = [
  86, 71, 93, 85, 79, 52, 69,
  20, 26, 79,  8, 30, 91, 98,
  18, 24, 82, 31, 92
]
partition[0,18,[86,71,93,85,79,52,69,20,26,79,8,30,91,98,18,24,82,31,92]]
partition[0,18,[86,71,85,79,52,69,20,26,79,8,30,91,18,24,82,31,92,98,93]] = 16
partition[0,15,[86,71,85,79,52,69,20,26,79,8,30,91,18,24,82,31,92,98,93]]
partition[0,15,[20,26,8,30,18,24,31,71,79,85,79,91,52,69,82,86,92,98,93]] = 6
partition[0,5,[20,26,8,30,18,24,31,71,79,85,79,91,52,69,82,86,92,98,93]]
partition[0,5,[20,8,18,24,26,30,31,71,79,85,79,91,52,69,82,86,92,98,93]] = 3
partition[0,2,[20,8,18,24,26,30,31,71,79,85,79,91,52,69,82,86,92,98,93]]
partition[0,2,[8,18,20,24,26,30,31,71,79,85,79,91,52,69,82,86,92,98,93]] = 1
partition[4,5,[8,18,20,24,26,30,31,71,79,85,79,91,52,69,82,86,92,98,93]]
partition[4,5,[8,18,20,24,26,30,31,71,79,85,79,91,52,69,82,86,92,98,93]] = 5
partition[7,15,[8,18,20,24,26,30,31,71,79,85,79,91,52,69,82,86,92,98,93]]
partition[7,15,[8,18,20,24,26,30,31,71,79,85,79,52,69,82,86,91,92,98,93]] = 14
par

In [ ]:
function isOrdered(L: number[]): void {
  for (let i = 0; i < L.length - 1; i++) {
    if (L[i] > L[i + 1]) {
      throw new Error(`${L} not ordered at ${i}`);
    }
  }
}

The function `counter` takes an array as input and returns a Map that keeps count of how many times each item occurs in the array.

In [15]:
function counter<T>(arr: T[]): Map<T, number> {
  const counts = new Map<T, number>();
  for (const item of arr) {
    counts.set(item, (counts.get(item) ?? 0) + 1);
  }
  return counts;
}

We also define the helper function `compareCounter` to be able to compare the contents of two counters.

In [16]:
function compareCounters(a: Map<number, number>, b: Map<number, number>): boolean {
  if (a.size !== b.size) return false;
  for (const [key, value] of a) {
    if (b.get(key) !== value) return false;
  }
  return true;
}

The function `sameElements(L, S)` returns `true` if the arrays `L` and `S` contain the same elements and, furthermore, each 
element $x$ occurring in `L` occurs in `S` the same number of times it occurs in `L`.

In [17]:
import assert from 'assert';

function sameElements(L: number[], S: number[]): void {
  assert(compareCounters(counter(L), counter(S)), "L and S do not have the same elements");
}

The function `randomIntRange(min, max)` generates a random integer in the range `[min, max)` and corresponds to Python's `range(min, max)` behavior regarding the exclusive upper bound.


In [18]:
function randomIntRange(min: number, max: number): number {
  return Math.floor(Math.random() * (max - min)) + min;
} 

The function $\texttt{testSort}(n, k)$ generates $n$ random arrays of length $k$, sorts them, and checks whether the output is sorted and contains the same elements as the input.

In [19]:
function testSort(n: number, k: number): void {
  for (let i = 0; i < n; i++) {
    const L = Array.from({ length: k }, () => randomIntRange(0, 2 * k));
    const oldL = [...L];
    sort(L);
    isOrdered(L);
    sameElements(oldL, L);
    console.assert(L.length === oldL.length, "Array length changed");
    process.stdout.write(".");
  }
  console.log("\nAll tests successful!");
}

In [20]:
console.time("testSort");
testSort(100, 20000);
console.timeEnd("testSort");

....................................................................................................
All tests successful!
testSort: 3.083s


Next, we sort a million random integers.

In [22]:
console.time("1 million random integers");
const k = 1_000_000;
const L1 = Array.from({ length: k }, () => randomIntRange(0, 1000));
sort(L1);
console.timeEnd("1 million random integers");

1 million random integers: 39.954s


Next, we sort a hundred thousand integers.  This time, many of the integers have the same value.

In [23]:
const L2 = Array.from({ length: 100_000 }, () => randomIntRange(0, 100));
console.time("100k integers (with repetitions)");
sort(L2);
console.timeEnd("100k integers (with repetitions)");

100k integers (with repetitions): 3.481s


Finally, we test the worst case and sort 5000 integers that are already sorted in ascending order. Since quicksort is recursive, this represents the worst possible recursion pattern.

In [24]:
const L3 = Array.from({ length: 5000 }, (_, i) => i);
console.time("worst case (sorted array)");
sort(L3);
console.timeEnd("worst case (sorted array)");

worst case (sorted array): 848.261ms


If we *shuffle* the array that is to be sorted before calling `sort`, the worst case behaviour disappears.

In [25]:
for (let i = L3.length - 1; i > 0; i--) {
  const j = Math.floor(Math.random() * (i + 1));
  [L3[i], L3[j]] = [L3[j], L3[i]];
}

In [26]:
console.time("shuffled array");
sort(L3);
console.timeEnd("shuffled array");

shuffled array: 2.473ms
